In [1]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio import Phylo
import os

# 1. Read in file
fasta_path = "/home3/oml4h/PLM_SARS-CoV-2/Sequences/HA_sequence_selection_incl_beijing.fas"
if not os.path.exists(fasta_path):
    # Try relative path if absolute fails
    fasta_path = "Sequences/HA_sequence_selection_incl_beijing.fas"

records = list(SeqIO.parse(fasta_path, "fasta"))



In [9]:
trimmed_records = []
for rec in records:

    # Ensure sequence is long enough
    if len(rec.seq) >= 566:
        trimmed_seq = rec.seq[1:566]

    
    trimmed_rec = SeqRecord(trimmed_seq, id=rec.id, description="")
    trimmed_records.append(trimmed_rec)
    
# 3. Make a tree
alignment = MultipleSeqAlignment(trimmed_records)
calculator = DistanceCalculator('identity')
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

# Draw the tree
print("Phylogenetic Tree (Full HA):")
Phylo.draw_ascii(tree)


Phylogenetic Tree (Full HA):
  , EPI2178977|HA|A/Thailand/8/2022|EPI_I...
 ,|
 || EPI2096148|HA|A/Massachusetts/18/2022...
 |
 , EPI_OMsynth|HA|A/OMsynth/2025|OM|J.2_int
 |
 | EPI2981619|HA|A/Croatia/10136RV/2023|...
 |
 |                  _______ tr|B2BV76|B2BV76_9INFA
 | ________________|
_||                |________________ sp|O11283|HEMA_I89A2
 ||
 ||  , weirdJ.2.2025
 || _|
 ||| | EPI4748783|HA|A/England/01837755/2025...
 |||
 ,|| EPI4551140|HA|A/England/415/2024|EPI_...
 |||
 ||| EPI4551140|HA|A/England/415/2024|EPI_...
 ||
 || EPI4551140|HA|A/England/415/2024|EPI_...
 |
 | EPI4551140|HA|A/England/415/2024|EPI_...



In [4]:

# 2. Trim to amino acids 17 to 345 (1-indexed)
# This corresponds to HA1 domain for H3N2
# 1-indexed 17 to 345 is 0-indexed 16 to 345 (exclusive)
trimmed_records = []
for rec in records:
    # Ensure sequence is long enough
    if len(rec.seq) >= 345:
        trimmed_seq = rec.seq[16:345]
    else:
        # If shorter, take what's available and pad with X if necessary
        # But based on previous check, they are at least 345
        trimmed_seq = rec.seq[16:345]
    
    trimmed_rec = SeqRecord(trimmed_seq, id=rec.id, description="")
    trimmed_records.append(trimmed_rec)

# 3. Make a tree
alignment = MultipleSeqAlignment(trimmed_records)
calculator = DistanceCalculator('identity')
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

# Draw the tree
print("Phylogenetic Tree (Full HA1, residues 17-345):")
Phylo.draw_ascii(tree)


Phylogenetic Tree (Full HA1, residues 17-345):
 , weirdJ.2.2025
 |
 |                ________ tr|B2BV76|B2BV76_9INFA
 | ______________|
 ||              |__________________ sp|O11283|HEMA_I89A2
 ||
 ,|  _ EPI4748783|HA|A/England/01837755/2025...
 || |
 ||_| EPI4551140|HA|A/England/415/2024|EPI_...
 || |
 || | EPI4551140|HA|A/England/415/2024|EPI_...
 ||
 || EPI4551140|HA|A/England/415/2024|EPI_...
 |
 | EPI4551140|HA|A/England/415/2024|EPI_...
 |
_|, EPI2178977|HA|A/Thailand/8/2022|EPI_I...
 ||
 || EPI2096148|HA|A/Massachusetts/18/2022...
 |
 | EPI2981619|HA|A/Croatia/10136RV/2023|...
 |
 | EPI_OMsynth|HA|A/OMsynth/2025|OM|J.2_int



In [5]:

# 4. Just take flu antigenic regions and make a tree
# Standard H3N2 antigenic sites (HA1 numbering, 1-indexed)
# Ref: Koel et al. 2013, Science and others
antigenic_sites = [
    # Site A
    122, 124, 126, 130, 131, 132, 133, 135, 137, 138, 140, 142, 143, 144, 145, 146, 150, 152, 167,
    # Site B
    128, 129, 155, 156, 157, 158, 159, 160, 163, 164, 165, 186, 187, 188, 189, 190, 192, 193, 194, 196, 197, 198,
    # Site C
    44, 45, 46, 47, 48, 50, 51, 53, 54, 273, 275, 276, 278, 279, 280, 294, 297, 299, 300, 304, 305, 307, 308, 309, 310, 311, 312,
    # Site D
    96, 102, 103, 117, 121, 167, 170, 171, 172, 173, 174, 175, 176, 177, 179, 182, 201, 203, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 226, 227, 228, 229, 230, 238, 240, 242, 244, 246, 247, 248,
    # Site E
    57, 59, 62, 63, 67, 75, 78, 80, 81, 82, 83, 86, 87, 88, 91, 92, 94, 109, 260, 261, 262, 265
]
# Remove duplicates and sort
antigenic_sites = sorted(list(set(antigenic_sites)))

# Extract antigenic sites (HA1 numbering is 1-indexed, so subtract 1 for index in trimmed sequence)
antigenic_records = []
for rec in trimmed_records:
    # rec.seq is the trimmed sequence (residues 17-345)
    # So index 0 in rec.seq is residue 17.
    # Residue x in HA1 numbering is index x-1 in rec.seq.
    antigenic_seq_str = "".join([str(rec.seq[site-1]) for site in antigenic_sites if site-1 < len(rec.seq)])
    antigenic_rec = SeqRecord(Seq(antigenic_seq_str), id=rec.id, description="")
    antigenic_records.append(antigenic_rec)

antigenic_alignment = MultipleSeqAlignment(antigenic_records)
antigenic_dm = calculator.get_distance(antigenic_alignment)
antigenic_tree = constructor.build_tree(antigenic_alignment)

print("\nPhylogenetic Tree (Antigenic Regions only):")
Phylo.draw_ascii(antigenic_tree)


Phylogenetic Tree (Antigenic Regions only):
  , EPI2178977|HA|A/Thailand/8/2022|EPI_I...
 ,|
 || EPI2096148|HA|A/Massachusetts/18/2022...
 |
 , EPI_OMsynth|HA|A/OMsynth/2025|OM|J.2_int
 |
 | EPI2981619|HA|A/Croatia/10136RV/2023|...
_|
 , weirdJ.2.2025
 |
 |                _________ tr|B2BV76|B2BV76_9INFA
 | ______________|
 ||              |__________________ sp|O11283|HEMA_I89A2
 ||
 ,| , EPI4748783|HA|A/England/01837755/2025...
 ||,|
 |||| EPI4551140|HA|A/England/415/2024|EPI_...
 |||
 ||| EPI4551140|HA|A/England/415/2024|EPI_...
 | |
 | | EPI4551140|HA|A/England/415/2024|EPI_...
 |
 | EPI4551140|HA|A/England/415/2024|EPI_...

